# 🚀 Model Serving — FastAPI + Docker

**Project 5 of 10 — MLOps Portfolio Series**  
**Author:** Jumma Mohammad Teli | Birmingham, UK  
**Dataset:** UCI Bank Marketing (41,188 rows)  
**Goal:** Deploy XGBoost champion from Project 4 as a production REST API

---

## Notebook Structure
1. Setup & Model Training
2. API Testing
3. Prediction Analysis
4. Batch Inference Demo
5. Key Takeaways

## 1. Setup & Model Training

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
print('✅ All imports successful')

In [ ]:
# Train champion model
from src.model.train import train_and_save

DATA_PATH = '../data/bank-additional-full.csv'
model, features, auc = train_and_save(
    data_path=DATA_PATH if os.path.exists(DATA_PATH) else None
)
print(f'\nModel trained | AUC: {auc:.4f} | Features: {len(features)}')

## 2. API Endpoints Demo

In [ ]:
from fastapi.testclient import TestClient
from src.api.app import app

client = TestClient(app)

# Health check
health = client.get('/health').json()
print('=== HEALTH CHECK ===')
for k, v in health.items():
    print(f'  {k}: {v}')

In [ ]:
# Model info
info = client.get('/model-info').json()
print('=== MODEL INFO ===')
print(f'  Model:     {info["model_type"]}')
print(f'  AUC:       {info["auc"]}')
print(f'  Features:  {info["n_features"]}')
print(f'  Baseline:  {info["baseline_auc"]} (Project 1)')

## 3. Single Prediction

In [ ]:
# Sample customer
CUSTOMER = {
    'age': 42, 'job': 'admin.', 'marital': 'married',
    'education': 'university.degree', 'default': 'no',
    'housing': 'yes', 'loan': 'no', 'contact': 'cellular',
    'month': 'may', 'day_of_week': 'mon', 'campaign': 2,
    'pdays': 999, 'previous': 0, 'poutcome': 'nonexistent',
    'emp.var.rate': 1.1, 'cons.price.idx': 93.994,
    'cons.conf.idx': -36.4, 'euribor3m': 4.857, 'nr.employed': 5191.0
}

result = client.post('/predict', json=CUSTOMER).json()
print('=== PREDICTION ===')
print(f'  Prediction:  {result["prediction"]} ({result["label"]})')
print(f'  Probability: {result["probability"]:.4f}')
print(f'  Confidence:  {result["confidence"]}')

## 4. Batch Inference Demo

In [ ]:
# Test different customer profiles
profiles = [
    {**CUSTOMER, 'age': 25, 'job': 'student', 'education': 'basic.4y'},
    {**CUSTOMER, 'age': 55, 'job': 'retired', 'poutcome': 'success'},
    {**CUSTOMER, 'age': 35, 'job': 'management', 'campaign': 8},
    {**CUSTOMER, 'age': 62, 'job': 'retired', 'previous': 3},
    {**CUSTOMER, 'age': 28, 'job': 'technician', 'housing': 'no'},
]

batch = client.post('/predict-batch', json={'customers': profiles}).json()

print(f'=== BATCH INFERENCE ===')
print(f'  Total customers:  {batch["total"]}')
print(f'  Will subscribe:   {batch["positive_count"]}')
print(f'  Positive rate:    {batch["positive_rate"]:.1%}')

for i, pred in enumerate(batch['predictions']):
    age = profiles[i]['age']
    job = profiles[i]['job']
    print(f'  {age}yr {job:<15}: {pred["label"]} ({pred["probability"]:.3f})')

In [ ]:
# Visualise batch results
ages = [p['age'] for p in profiles]
probs = [p['probability'] for p in batch['predictions']]
jobs = [p['job'] for p in profiles]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#DC2626' if p > 0.5 else '#2563EB' for p in probs]
axes[0].barh(jobs, probs, color=colors, edgecolor='white')
axes[0].axvline(x=0.5, color='black', linestyle='--', lw=1.5, label='Decision boundary')
axes[0].set_xlabel('Subscription Probability')
axes[0].set_title('Prediction Probabilities by Job', fontweight='bold')
axes[0].legend()

axes[1].scatter(ages, probs, c=colors, s=100, edgecolor='white', linewidth=1.5)
axes[1].axhline(y=0.5, color='black', linestyle='--', lw=1.5)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Subscription Probability')
axes[1].set_title('Probability by Age', fontweight='bold')

plt.suptitle('Batch Inference Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Key Takeaways

In [ ]:
print('=' * 60)
print('KEY TAKEAWAYS — Model Serving')
print('=' * 60)
print('''
API ENDPOINTS
  GET  /health       — Health check
  GET  /model-info   — Model metadata
  POST /predict      — Single prediction
  POST /predict-batch — Batch predictions

MODEL
  XGBoost (tuned with Optuna — Project 4)
  AUC: 0.8176 on real UCI Bank Marketing data
  19 features (duration dropped — leakage)

INFRASTRUCTURE
  FastAPI + Pydantic validation
  Docker multi-stage build
  Non-root user security
  Health check endpoint
  CI/CD: train -> test -> docker build

NEXT STEPS
  Deploy to Google Cloud Run (free tier)
  Project 6: Feature Store with Feast + Redis
  Project 7: Monitor live predictions for drift
''')
print('=' * 60)

---
**Author:** Jumma Mohammad Teli | Birmingham, UK  
**LinkedIn:** https://linkedin.com/in/jumma-mohammad  
**GitHub:** https://github.com/jumma786  

*Project 5 of 10 — MLOps Portfolio Series*